## Prepare for analysis - get models

**Load modules**

In [1]:
import reframed
import pandas as pd
import warnings
import cobra

import sys

sys.path.append("../functions/")
from minimal_medium import minimal_medium

sys.path.append("../scripts_python/Venturelli2018/")

import prepare_models

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
prepare_models.main()

INFO     | LOAD DATA AND MODELS
INFO     | load media data and models from '../models/Venturelli2018/species_used/'...
/Users/idunmariaburgos/SteadierCom/reframed/reframed/io/sbml.py:579: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C12H21N3O13P2/c13-8-1-2-15(12(20)14-8)11-10(19)9(18)7(27-11)5-26-30(23,24)28-29(21,22)25-4-6(17)3-16/h1-2,6-7,9-11,16-19H,3-5H2,(H,21,22)(H,23,24)(H2,13,14,20)/
  warn(f'Could not extract annotation from {term.getResourceURI(i)}')
/Users/idunmariaburgos/SteadierCom/reframed/reframed/io/sbml.py:579: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C11H12Cl2N2O5/c12-10(13)11(18)14-8(5-16)9(17)6-1-3-7(4-2-6)15(19)20/h1-4,8-10,16-17H,5H2,(H,14,18)/
  warn(f'Could not extract annotation from {term.getResourceURI(i)}')
/Users/idunmariaburgos/SteadierCom/reframed/reframed/io/sbml.py:579: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C15H26N2O12/c1-4(2

In [6]:
rxns_members = atp_EGC_community(models)


R_DM_atp_c__M_Bacteroides_uniformis_ATCC_8492: 1000.0
R_DM_atp_c__M_Bacteroides_vulgatus_ATCC_8482: 1000.0
R_DM_atp_c__M_Collinsella_aerofaciens_ATCC_25986: 1000.0
R_DM_atp_c__M_Faecalibacterium_prausnitzii_A2_165: 1000.0
R_DM_atp_c__M_Eggerthella_lenta_DSM_2243: 1000.0
R_DM_atp_c__M_Prevotella_copri_CB7_DSM_18205: 1000.0
R_DM_atp_c__M_Clostridium_hiranonis_TO_931_DSM_13275: 1000.0
R_DM_atp_c__M_Eubacterium_rectale_ATCC_33656: 1000.0
R_DM_atp_c__M_Desulfovibrio_piger_ATCC_29098: 1000.0
R_DM_atp_c__M_Bacteroides_ovatus_ATCC_8483: 1000.0
R_DM_atp_c__M_Bacteroides_thetaiotaomicron_VPI_5482: 1000.0
R_DM_atp_c__M_Blautia_hydrogenotrophica_DSM_10507: 1000.0


In [ ]:
transport_results = find_transport_mets_comm(models,rxns_members)

In [154]:
# Make df from transporter information
df1 = pd.DataFrame(transport_results).sort_values("met")
df2 = df1.pivot_table(index="rxn",columns="met",values="member",aggfunc=(lambda x: ", ".join(x)))

# Find compounds that have different transporters
many_transporters = df2.count()[df2.count()>1].index
many_transporters = [met for met in many_transporters if met!="M_h__91__e__93__"]

# Filter df for these compounds
df3 = df1[df1.met.isin(many_transporters)].copy()
df3["ATPM"]=rxn
different_transporter.append(df3)
df4 = pd.concat(different_transporter)
df5 = df4[["rxn","met","equation"]].drop_duplicates()
df6 = df4.loc[df5.index,["member","rxn","met"]].copy()

print("FIND PROBLEMATIC METS: df6 has all mets that have more than 1 rxn for transport. ")
# We need to find out if this is creating problems or not.
# Finding reactions that are different but transport the same compound
problematic_mets = []
for met in df6.met.unique():
    df6_slice = df6[df6.met==met]
    
    problematic = True
    
    reactions = []
    order = True
    for member in df6_slice.member.values:    
        # find reaction
        rxn = df6_slice.query(f"member=='{member}'").rxn.iloc[0]
        substrates=models[member[2:]].reactions[rxn].get_substrates()
        products=models[member[2:]].reactions[rxn].get_products()
        
        dict_ = {"rxn":rxn,
                          "set1":tuple(sorted(substrates)),
                          "set2":tuple(sorted(products)),
                          }
        if order:
            dict_["set3"]=tuple(sorted(substrates))
            dict_["set4"]=tuple(sorted(products))
            order=False
        else:
            dict_["set3"]=tuple(sorted(products))
            dict_["set4"]=tuple(sorted(substrates))
            order=True
        reactions.append(dict_)
    test = pd.DataFrame(reactions)
    
    for set_ in ["set1","set2","set3","set4"]:
        len_unique = len(test[set_].unique())
        
        if len_unique==1:
            problematic=False
            break
            
    if problematic:
        problematic_mets.append(met)

print("These metabolites have multiple transport reactions which could contribute to energy generating cycles:")
print(problematic_mets)

In [154]:
comm_test = reframed.Community('transporter_test',models_new.values())
comm_model = comm_test.merged_model
env_empty = reframed.Environment.empty(comm_model)
env_empty.apply(comm_model,inplace=True)

for member in comm_test.organisms.keys():
    rxn = "R_DM_atp_c__"+member
    comm_model.reactions[rxn].lb=0
    comm_model.reactions[rxn].ub=1000

transport_rxns = comm_model.get_reactions_by_type(reframed.ReactionType.TRANSPORT)

sol_fva = reframed.FVA(comm_model,reactions=transport_rxns)

for key,values in sol_fva.items():
    if abs(values[0])>1e-6 or abs(values[1])>1e-6:
        print(f"{key}: {values}")
        

### Check mass balance with cobra

In [229]:
import cobra

In [238]:
model = cobra.io.read_sbml_model("../models/Venturelli2018/species_used_noO2/Bacteroides_ovatus_ATCC_8483.xml")

Adding exchange reaction EX_12ppd_S[e] with default bounds for boundary metabolite: 12ppd_S[e].
Adding exchange reaction EX_1hibup_S[e] with default bounds for boundary metabolite: 1hibup_S[e].
Adding exchange reaction EX_1hibupglu_S[e] with default bounds for boundary metabolite: 1hibupglu_S[e].
Adding exchange reaction EX_1hmdgluc[e] with default bounds for boundary metabolite: 1hmdgluc[e].
Adding exchange reaction EX_1ohmdz[e] with default bounds for boundary metabolite: 1ohmdz[e].
Adding exchange reaction EX_2hatvacid[e] with default bounds for boundary metabolite: 2hatvacid[e].
Adding exchange reaction EX_2hatvacidgluc[e] with default bounds for boundary metabolite: 2hatvacidgluc[e].
Adding exchange reaction EX_2hatvlac[e] with default bounds for boundary metabolite: 2hatvlac[e].
Adding exchange reaction EX_2hatvlacgluc[e] with default bounds for boundary metabolite: 2hatvlacgluc[e].
Adding exchange reaction EX_2hibup_S[e] with default bounds for boundary metabolite: 2hibup_S[e].


In [254]:
for reaction in model.reactions:
    balance = reaction.check_mass_balance()
    
    if len(balance)>0 and not reaction.id.startswith("EX_"):
        print(f"{reaction}: {balance}")

DHNAOPT: dhna[c] + phyldpp[c] <=> co2[c] + dmphqno[c] + h[c] + ppi[c]: {'charge': 2.0}
DHNAOT: dhna[c] + octdp[c] --> 2dmmq8[c] + co2[c] + h[c] + ppi[c]: {'charge': 2.0}
DM_2HYMEPH: 2hymeph[c] --> : {'C': -7.0, 'H': -8.0, 'O': -2.0}
DM_4HBA: 4hba[c] --> : {'C': -7.0, 'H': -8.0, 'O': -2.0}
DM_AMOB: amob[c] --> : {'C': -15.0, 'H': -19.0, 'N': -5.0, 'O': -6.0, 'S': -1.0}
DM_adocbl(c): adocbl[c] --> : {'C': -72.0, 'H': -100.0, 'Co': -1.0, 'N': -18.0, 'O': -17.0, 'P': -1.0}
DM_btn: btn[c] --> : {'charge': 1.0, 'C': -10.0, 'H': -15.0, 'N': -2.0, 'O': -3.0, 'S': -1.0}
DM_dad_5: dad_5[c] --> : {'C': -10.0, 'H': -13.0, 'N': -5.0, 'O': -3.0}
DM_kdo2lipid4L(c): kdo2lipid4L[c] --> : {'charge': 6.0, 'C': -96.0, 'H': -170.0, 'N': -2.0, 'O': -38.0, 'P': -2.0}
DM_pheme(c): pheme[c] --> : {'charge': 2.0, 'C': -34.0, 'H': -30.0, 'Fe': -1.0, 'N': -4.0, 'O': -4.0}
DM_this[c]: this[c] --> : {'charge': 1.0, 'R': -1.0, 'C': -1.0, 'O': -2.0}
GLCP3: glycogen[c] + h[c] + pi[c] <=> g1p[c] + glycogenb[c]: {'charg